In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/clean_matches.csv')
df = df.sort_values('Year').reset_index(drop=True)

print(f"Loaded: {df.shape}")
print(f"Years: {df['Year'].min()} → {df['Year'].max()}")
print(f"Sample winners: {df['Winner'].value_counts().head()}")

Loaded: (980, 21)
Years: 1930 → 2022
Sample winners: Winner
Draw         220
Brazil        77
Argentina     49
Italy         45
France        40
Name: count, dtype: int64


In [15]:
stage_mapping = {
    'Group 1': 1, 'Group 2': 1, 'Group 3': 1,
    'Group 4': 1, 'Group 5': 1, 'Group 6': 1,
    'Group A': 1, 'Group B': 1, 'Group C': 1,
    'Group D': 1, 'Group E': 1, 'Group F': 1,
    'Group G': 1, 'Group H': 1,
    'Preliminary round': 1, 'First round': 1,
    'Round of 16': 2, 'Quarter-finals': 3,
    'Semi-finals': 4,
    'Third place': 5,
    'Match for third place': 5,
    'Play-off for third place': 5,
    'Final': 6
}

df['stage_encoded'] = df['Stage'].map(stage_mapping).fillna(1)

missing = df[df['stage_encoded'].isnull()]['Stage'].unique()
print(f"Unmapped stages: {missing}")
print(f"Stage distribution:\n{df['stage_encoded'].value_counts().sort_index()}")

Unmapped stages: <ArrowStringArray>
[]
Length: 0, dtype: str
Stage distribution:
stage_encoded
1    735
2     88
3     74
4     40
5     21
6     22
Name: count, dtype: int64


In [16]:
# Starting Elo for all teams
BASE_ELO = 1000
K_FACTOR = 20  # How much each match shifts ratings

elo_ratings = {}

def get_elo(team):
    return elo_ratings.get(team, BASE_ELO)

def expected_score(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def update_elo(winner, loser, draw=False):
    elo_w = get_elo(winner)
    elo_l = get_elo(loser)
    exp_w = expected_score(elo_w, elo_l)
    
    if draw:
        actual_w = 0.5
        actual_l = 0.5
    else:
        actual_w = 1.0
        actual_l = 0.0
    
    elo_ratings[winner] = elo_w + K_FACTOR * (actual_w - exp_w)
    elo_ratings[loser]  = elo_l + K_FACTOR * (actual_l - (1 - exp_w))

# Calculate Elo for every match
home_elos, away_elos = [], []

for _, row in df.iterrows():
    ht = row['Home Team Name']
    at = row['Away Team Name']
    
    # Record Elo BEFORE this match
    home_elos.append(get_elo(ht))
    away_elos.append(get_elo(at))
    
    # Update Elo AFTER this match
    if row['Winner'] == ht:
        update_elo(ht, at)
    elif row['Winner'] == at:
        update_elo(at, ht)
    else:
        update_elo(ht, at, draw=True)

df['home_elo'] = home_elos
df['away_elo'] = away_elos
df['elo_diff'] = df['home_elo'] - df['away_elo']

print("Elo ratings calculated ✅")
print(f"\nTop 10 teams by final Elo:")
sorted_elo = sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)
for team, elo in sorted_elo[:10]:
    print(f"  {team}: {elo:.0f}")

Elo ratings calculated ✅

Top 10 teams by final Elo:
  Brazil: 1167
  Netherlands: 1164
  Germany: 1160
  France: 1148
  Argentina: 1133
  Germany FR: 1129
  Italy: 1113
  Spain: 1078
  England: 1068
  Belgium: 1053


In [17]:
def get_attack(df, team, before_year):
    past = df[df['Year'] < before_year]
    home_goals = past[past['Home Team Name'] == team]['Home Team Goals']
    away_goals = past[past['Away Team Name'] == team]['Away Team Goals']
    all_goals = pd.concat([home_goals, away_goals])
    return all_goals.mean() if len(all_goals) >= 3 else 1.0

def get_defense(df, team, before_year):
    past = df[df['Year'] < before_year]
    home_conceded = past[past['Home Team Name'] == team]['Away Team Goals']
    away_conceded = past[past['Away Team Name'] == team]['Home Team Goals']
    all_conceded = pd.concat([home_conceded, away_conceded])
    return all_conceded.mean() if len(all_conceded) >= 3 else 1.0

def get_win_rate(df, team, before_year):
    past = df[df['Year'] < before_year]
    home = past[past['Home Team Name'] == team]
    away = past[past['Away Team Name'] == team]
    total = len(home) + len(away)
    if total == 0:
        return 0.5
    wins = len(past[past['Winner'] == team])
    return wins / total

print("Building attack, defense, win rate features...")
print("This takes 1-2 minutes — please wait...")

home_att, away_att = [], []
home_def, away_def = [], []
home_wr, away_wr   = [], []

for _, row in df.iterrows():
    yr = row['Year']
    ht = row['Home Team Name']
    at = row['Away Team Name']
    
    home_att.append(get_attack(df, ht, yr))
    away_att.append(get_attack(df, at, yr))
    home_def.append(get_defense(df, ht, yr))
    away_def.append(get_defense(df, at, yr))
    home_wr.append(get_win_rate(df, ht, yr))
    away_wr.append(get_win_rate(df, at, yr))

df['home_attack']    = home_att
df['away_attack']    = away_att
df['home_defense']   = home_def
df['away_defense']   = away_def
df['home_goal_diff'] = df['home_attack'] - df['home_defense']
df['away_goal_diff'] = df['away_attack'] - df['away_defense']
df['home_win_rate']  = home_wr
df['away_win_rate']  = away_wr

print("Done ✅")

Building attack, defense, win rate features...
This takes 1-2 minutes — please wait...
Done ✅


In [18]:
def get_weighted_form(df, team, before_year, last_n=5):
    past = df[df['Year'] < before_year]
    matches = past[
        (past['Home Team Name'] == team) |
        (past['Away Team Name'] == team)
    ].tail(last_n)
    
    if len(matches) == 0:
        return 0.5
    
    # More recent matches weighted more heavily
    weights = np.array([0.1, 0.15, 0.2, 0.25, 0.3])
    weights = weights[-len(matches):]
    weights = weights / weights.sum()
    
    scores = []
    for _, m in matches.iterrows():
        if m['Winner'] == team:
            scores.append(1.0)
        elif m['Winner'] == 'Draw':
            scores.append(0.5)
        else:
            scores.append(0.0)
    
    return float(np.dot(scores, weights))

def get_head_to_head(df, home, away, before_year):
    past = df[df['Year'] < before_year]
    h2h = past[
        ((past['Home Team Name'] == home) & (past['Away Team Name'] == away)) |
        ((past['Home Team Name'] == away) & (past['Away Team Name'] == home))
    ]
    if len(h2h) == 0:
        return 0.5
    home_wins = len(h2h[h2h['Winner'] == home])
    return home_wins / len(h2h)

print("Building form and head-to-head features...")
print("This takes 1-2 minutes — please wait...")

home_form, away_form, h2h_list = [], [], []

for _, row in df.iterrows():
    yr = row['Year']
    ht = row['Home Team Name']
    at = row['Away Team Name']
    
    home_form.append(get_weighted_form(df, ht, yr))
    away_form.append(get_weighted_form(df, at, yr))
    h2h_list.append(get_head_to_head(df, ht, at, yr))

df['home_recent_form'] = home_form
df['away_recent_form'] = away_form
df['head_to_head']     = h2h_list

print("Done ✅")

Building form and head-to-head features...
This takes 1-2 minutes — please wait...
Done ✅


In [21]:
# Difference features — relative strength matters more than absolutes
df['attack_diff']    = df['home_attack']    - df['away_attack']
df['defense_diff']   = df['away_defense']   - df['home_defense']
df['form_diff']      = df['home_recent_form'] - df['away_recent_form']
df['goal_diff_diff'] = df['home_goal_diff'] - df['away_goal_diff']
df['elo_ratio']      = df['home_elo'] / df['away_elo'].replace(0, 1)

print("Difference features added ✅")
print(df[['attack_diff','defense_diff','form_diff',
          'goal_diff_diff','elo_ratio']].describe().round(3))

Difference features added ✅
       attack_diff  defense_diff  form_diff  goal_diff_diff  elo_ratio
count      980.000       980.000    980.000         980.000    980.000
mean         0.191         0.107      0.057           0.298      1.025
std          0.880         0.820      0.303           1.229      0.076
min         -3.273        -4.146     -1.000          -5.490      0.779
25%         -0.333        -0.333     -0.125          -0.418      0.980
50%          0.226         0.039      0.025           0.250      1.015
75%          0.709         0.594      0.250           1.062      1.067
max          3.273         3.333      1.000           5.500      1.261


In [22]:
def encode_result(row):
    if row['Winner'] == row['Home Team Name']:
        return 1   # Team A wins
    elif row['Winner'] == row['Away Team Name']:
        return 2   # Team B wins
    else:
        return 0   # Draw

df['result'] = df.apply(encode_result, axis=1)

features = [
    # Tier 1 — Absolute strength
    'home_elo', 'away_elo', 'elo_diff', 'elo_ratio',
    'home_attack', 'away_attack',
    'home_defense', 'away_defense',
    'home_goal_diff', 'away_goal_diff',
    # Tier 1B — Relative strength (new)
    'attack_diff', 'defense_diff',
    'goal_diff_diff',
    # Tier 2 — Form
    'home_recent_form', 'away_recent_form',
    'home_win_rate', 'away_win_rate',
    'form_diff',
    # Tier 3 — Context
    'stage_encoded',
    # Target
    'result'
]

df_model = df[features].copy()

print(f"Final dataset shape: {df_model.shape}")
print(f"\n{len(features)-1} features + 1 target")
print(f"\nFeature list:")
for f in features[:-1]:
    print(f"  → {f}")

print(f"\nResult distribution:")
print(df_model['result'].value_counts())

df_model.to_csv('../data/processed/model_features.csv', index=False)
print("\nSaved to data/processed/model_features.csv ✅")

Final dataset shape: (980, 20)

19 features + 1 target

Feature list:
  → home_elo
  → away_elo
  → elo_diff
  → elo_ratio
  → home_attack
  → away_attack
  → home_defense
  → away_defense
  → home_goal_diff
  → away_goal_diff
  → attack_diff
  → defense_diff
  → goal_diff_diff
  → home_recent_form
  → away_recent_form
  → home_win_rate
  → away_win_rate
  → form_diff
  → stage_encoded

Result distribution:
result
1    547
0    220
2    213
Name: count, dtype: int64

Saved to data/processed/model_features.csv ✅
